# Parameter Golf - Google Colab

OpenAI Model Craft Challenge: Parameter Golf の検証用ノートブック。

**ランタイム設定**: 上メニューから「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** を選択してください。

## 1. GPU確認

In [ ]:
!nvidia-smi

## 2. リポジトリのクローンと依存関係インストール

In [ ]:
!git clone https://github.com/tsubasagit/parameter-golf.git /content/parameter-golf
%cd /content/parameter-golf
!pip install -q sentencepiece huggingface-hub datasets tqdm

## 3. データセットのダウンロード（最小構成: 1 shard）

In [ ]:
!python3 data/cached_challenge_fineweb.py --variant sp1024 --train-shards 1

## 4. トレーニング実行（ベースライン）

Colab T4 は H100 より遅いため、イテレーション数を減らして検証します。

In [ ]:
import os
os.environ['RUN_ID'] = 'colab_baseline'
os.environ['ITERATIONS'] = '500'
os.environ['TRAIN_BATCH_TOKENS'] = '131072'
os.environ['VAL_LOSS_EVERY'] = '100'
os.environ['MAX_WALLCLOCK_SECONDS'] = '600'

!torchrun --standalone --nproc_per_node=1 train_gpt.py

## 5. 結果確認

上のセルの最後に `val_loss`, `val_bpb`, 圧縮モデルサイズが表示されます。

- ベースライン目標: val_bpb ≈ 1.22
- SOTA (8xH100): val_bpb = 1.1428

T4 + 500イテレーションではベースラインには届きませんが、パイプラインの動作確認とアーキテクチャ実験には十分です。

## 6. カスタム実験（ここを編集）

ハイパーパラメータを変えて再実行できます。

In [ ]:
import os
os.environ['RUN_ID'] = 'colab_experiment_1'
os.environ['ITERATIONS'] = '500'
os.environ['TRAIN_BATCH_TOKENS'] = '131072'
os.environ['VAL_LOSS_EVERY'] = '100'
os.environ['MAX_WALLCLOCK_SECONDS'] = '600'

# --- ここでパラメータを調整 ---
# os.environ['NUM_LAYERS'] = '10'
# os.environ['MODEL_DIM'] = '512'
# os.environ['NUM_HEADS'] = '8'
# os.environ['MLP_MULT'] = '3'

!torchrun --standalone --nproc_per_node=1 train_gpt.py